Note: we have anonymised this repo so this notebook might not work very well. We will release the fully working version upon acceptance

In [ ]:
import json
import os
import pandas as pd
import ast

from sklearn.model_selection import train_test_split

In [ ]:
dialect_syn = 'nl-FY'
dialect = 'west_frisian'

real_data_file = f'data/data/human_annotated/{dialect}_dataset_with_rubric_shuffled.json'
synth_data_file = f'data/data/synthetic_data/{dialect_syn}_dataset_synthetic_corrected.json'

In [7]:
with open(real_data_file, 'r', encoding='utf-8') as f:
    real = json.load(f)
with open(synth_data_file, 'r', encoding='utf-8') as f:
    synth = json.load(f)

In [8]:
len(real), len(synth)

(1015, 5973)

### get real data train and test

In [9]:
labels = [item['Label'] for item in real]
train_set, test_set = train_test_split(real, test_size=0.2, random_state=42, stratify=labels)

In [ ]:
with open(f"data/finetuning_data/processed/{dialect}_real.json", 'w', encoding='utf-8') as f:
    json.dump(train_set, f, ensure_ascii=False, indent=2)
with open(f"data/finetuning_data/processed/{dialect}_test.json", 'w', encoding='utf-8') as f:
    json.dump(test_set, f, ensure_ascii=False, indent=2)

### add synthetic data to train data

In [ ]:
def dedup_by_prompt_output(items):
    seen, deduped = set(), []
    for it in items:
        key = (it.get("Prompt"), it.get("Output"))
        if key in seen:
            continue
        seen.add(key)
        deduped.append(it)
    return deduped

for crit in criteria_keys:
    synth_neg = [s for s in synth if s.get("Rubric", {}).get(crit) == 0]
    combined = dedup_by_prompt_output(real + synth_neg)
    out_path = f"data/finetuning_data/processed/{dialect}_synth_{crit}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)
    print(f"{crit}: real={len(real)}, synth_neg={len(synth_neg)}, combined={len(combined)} -> {out_path}")

c1: real=1015, synth_neg=995, combined=2010 -> data/processed/west_frisian_synth_c1.json
c2a: real=1015, synth_neg=998, combined=2013 -> data/processed/west_frisian_synth_c2a.json
c2b: real=1015, synth_neg=997, combined=2012 -> data/processed/west_frisian_synth_c2b.json
c3: real=1015, synth_neg=994, combined=2009 -> data/processed/west_frisian_synth_c3.json
c4: real=1015, synth_neg=993, combined=2008 -> data/processed/west_frisian_synth_c4.json
c5: real=1015, synth_neg=996, combined=2011 -> data/processed/west_frisian_synth_c5.json


### get synthetic data only

In [ ]:
with open(f'data/finetuning_data/processed/{dialect}_syn_all.json', 'w', encoding='utf-8') as f:
    json.dump(synth, f, ensure_ascii=False, indent=2)

In [ ]:
# syn_real = dedup_by_prompt_output(real + synth)
# with open(f'data/finetuning_data/processed/{dialect}_syn_real.json', 'w', encoding='utf-8') as f:
#     json.dump(syn_real, f, ensure_ascii=False, indent=2)